## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading/loading the **FEI Morph v2** and the **FEI Face** datasets.

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
print("Starting the download of the FEI Morph dataset...")
print("Note: This might take a while the first time...")

# Raw FEI datasets live in the shared Utility/ folder (same place as Utility/FaceDetection/,
# used later in this notebook), not next to this notebook.
UTILITY_DIR = Path.cwd().parent.parent / "Utility"  # Datasets/ -> 02_Extended_Framework/ -> deepfake-forensics-pipeline/Utility

fei_morph_path = UTILITY_DIR / "FEI Morph v2"
print("Dataset path:", fei_morph_path) if fei_morph_path.is_dir() else print("Dataset not founded")

fei_face_path = UTILITY_DIR / "FEI Face"
print("Dataset path:", fei_face_path if fei_face_path.is_dir() else print("Dataset not founded"))

In [ ]:
folders = os.listdir(fei_morph_path)
tot_elements = len(list(fei_morph_path.iterdir()))
print(f"Total elements inside { fei_morph_path}: {tot_elements}")
print("Contents inside", fei_morph_path, ": ")
print('\n'.join([f"- {f}" for f in folders[:10]]))

In [ ]:
folders = os.listdir(fei_face_path)
print("Contents inside", fei_face_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))

In [ ]:
subfolder_name = folders[-1]
subfolder_path = os.path.join(fei_face_path, subfolder_name)
if os.path.isdir(subfolder_path):
    contents = os.listdir(subfolder_path)
    print(f"Contents inside the first subfolder: {subfolder_name}")
    print('\n'.join([f"- {f}" for f in contents[:10]]))

## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all images from the datasets.

Columns:
- `filename`: filename
- `path`: file path 
- `label`: 0 = real/original, 1 = fake/morphed
- `subj1`: ID of the first subject 
- `subj2`: ID of the second subject; NaN if the image is an original
- `pose_number`: Pose identifier for Subject 1 
- `pose_number_2`: Pose identifier for Subject 2 
- `algorithm`: morphing algorithm used or "original"
- `morph_factor_1`: blending percentage of Subject 1 
- `morph_factor_2`: blending percentage of Subject 2 
- `post_proc_auto`: automated post-processing 
- `post_proc_manual`: manual retouching or professional editing.
- `digital_ps`: digital alterations or Photoshop-style enhancements.
- `resolution`: image dimensions

In [ ]:
def get_resolution(file_path):
    with Image.open(file_path) as img:
        width, height = img.size
        resolution = f"{width}x{height}"
    return resolution

morph_data = []

for file_path in fei_morph_path.glob("*.png"):
    parts = file_path.stem.split("_")

    if len(parts) == 9:
        
        resolution = get_resolution(file_path)

        subj1_raw = parts[1]
        subj2_raw = parts[2]
        
        subj1 = subj1_raw.split("-")[0] if "-" in subj1_raw else subj1_raw
        pose_num = subj1_raw.split("-")[1] if "-" in subj1_raw else None
        
        subj2 = subj2_raw.split("-")[0] if "-" in subj2_raw else subj2_raw
        pose_num_2 = subj2_raw.split("-")[1] if "-" in subj2_raw else None
                
        morph_data.append({
            "filename": file_path.name,
            "path": str(file_path),
            "label": 1,
            "subj1": int(subj1),
            "subj2": int(subj2),
            "pose_number": pose_num,
            "pose_number_2": pose_num_2,
            "algorithm": parts[3],
            "morph_factor_1": parts[4],
            "morph_factor_2": parts[5],
            "post_proc_auto": parts[6],
            "post_proc_manual": parts[7],
            "digital_ps": parts[8],
            "resolution": resolution
        })
    
real_data = []

for file_path in fei_face_path.rglob("*.jpg"):

    parts = file_path.stem.split("-")
    
    if len(parts) == 2:
        subj_id = parts[0]
        pose_num = parts[1]
    

        resolution = get_resolution(file_path)

        real_data.append({
            "filename": file_path.name,
            "path": str(file_path),
            "label": 0,
            "subj1": int(subj_id),
            "subj2": pd.NA,
            "pose_number": pose_num,
            "pose_number_2": None,
            "algorithm": "original",
            "morph_factor_1": None,
            "morph_factor_2": None,
            "post_proc_auto": None,
            "post_proc_manual": None,
            "digital_ps": None,
            "resolution": resolution
        })

morphed = pd.DataFrame(morph_data)
originals = pd.DataFrame(real_data)

pd.set_option('display.max_colwidth', None)
display(morphed.sample(20))
display(originals.head())

In [ ]:
print("--- POSE BALANCING ---")

print(f"Originals before filtering: {len(originals)}") #(Should be 2800)
originals = originals[originals['pose_number'] == '11']

print(f"Originals after filtering: {len(originals)}") #(Should be 200, one per subject)

full_fei_df = pd.concat([morphed, originals], ignore_index=True)

print(f"\nDataset MERGED and BALANCED successfully!")
print(f"Total images: {len(full_fei_df)} (Morphed: {len(morphed)} + Originals: {len(originals)})") #(Should be 14200, one per subject)

### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly:

* **Total number of images**
* **Label distribution (Fake vs Original)**: analysis of the class balance
* **Morphing Method Distribution**: breakdown of images by the algorithm used
* **Identity Consistency Check**: verification that `subj1` and `subj2` are always different in morphing rows to ensure no "self-morphs" (which would technically be originals) exist in the fake category
* **Unique Identity Count**: identification of all unique individuals (200 in total) involved in the dataset
* **Missing Values Check**: validation of the DataFrame's integrity. We expect `subj2` to have null values for original images, but other critical columns (like `path` or `label`) must be fully populated
* **Post-processing Automation (PA) Levels**: verification of the distribution of automated enhancements applied to the images, which can range from no processing (PA00) to various levels of digital adjustment

In [ ]:
print("--- SANITY CHECK ON FULL DATASET ---")

print(f"Total images: {len(full_fei_df)}")
print(f"\nLabel distribution (Fake vs Original):\n{full_fei_df['label'].value_counts()}")

print(f"\nMorphing Method (Algorithm) distribution:\n{full_fei_df['algorithm'].value_counts(dropna=False)}")

fakes_only = full_fei_df[full_fei_df['label'] == 'fake']
identities_match = (fakes_only['subj1'] == fakes_only['subj2']).any()
print(f"\nAre there any cases where Subject 1 == Subject 2 in morphs? {'YES' if identities_match else 'NO'}")

all_identities = set(full_fei_df['subj1'].dropna()).union(set(full_fei_df['subj2'].dropna()))
print(f"\nTotal unique individuals in the dataset: {len(all_identities)}")
print(f"Unique identities (Source 1): {full_fei_df['subj1'].nunique()}")
print(f"Unique identities (Source 2): {full_fei_df['subj2'].nunique()}")

print(f"\nChecking for missing values:\n{full_fei_df.isnull().sum()}")

print(f"\nPost-processing automation levels (PA):\n{full_fei_df['post_proc_auto'].value_counts(dropna=False)}")

### 2.2  - Distribution of Fake Images per Subject (Subj1)

Analyzes how many fake images exist per subj1 to identify if some identities dominate the fake samples

In [ ]:
fake_df = full_fei_df[full_fei_df["label"] == 1]
per_target = fake_df.groupby("subj1").size()
print("\n--- FAKE PER SUBJ1 STATISTICS ---")
print(per_target.describe())

### 2.3 - Distribution Of Methods per Subject (Subj1) (Check Variability):

Creates a pivot table showing how many images of each algorithm exist per subj1 to verify that each identity has a representative set of algorithm and to detect identities with too few or missing algorithm 
types

In [ ]:
pivot = pd.pivot_table(
    full_fei_df,
    index="subj1",
    columns="algorithm",
    values="filename",
    aggfunc="count",
    fill_value=0
)

print("\n--- ALGORITHM DISTRIBUTION PER SUB1 ---")
display(pivot.sample(20))

### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset to understand dataset imbalance

In [ ]:
print("Normalize label ditribution:")
print(full_fei_df["label"].value_counts(normalize=True))

### Dataset Observations

Based on the sanity check results, several key characteristics of the merged FEI Morph dataset emerge.

* **Extreme Class Imbalance**: The dataset is heavily skewed towards morphed images. There are **14,000 Fakes (98.6%)** compared to only **200 Originals (1.4%)**.
* **Perfect Algorithm & Pipeline Symmetry**: The 14,000 morphs are perfectly distributed across 7 different morphing algorithms (`C01`, `C02`, `C03`, `C05`, `C08`, `C15`, `C16`), with exactly **2,000 images per algorithm**. Additionally, the Post-Processing Automation (PA) levels map 1:1 with these algorithms.
* **Logical Missing Values**: There are exactly 200 missing values in columns like `subj2`, `morph_factor`, and `post_proc_auto`. This confirms perfect data integrity, as these 200 rows correspond exactly to the 200 "Original" images (which naturally do not have a second subject or morphing parameters).
* **Identity Integrity**: There are 200 unique individuals in the dataset, and the check confirms there are **zero instances of self-morphing** (`Subject 1 == Subject 2` is False across the board).
* **Subject Variance in Morphs**: While the overall algorithms are balanced, the frequency of a specific person acting as the base face (`subj1`) varies significantly. Some subjects act as the base for up to **252 morphs**, while others are used only **14 times** (mean $\approx$ 71). 
* **Intra-Subject Algorithm Balance**: Despite the variance in how often a subject is used as `subj1`, the distribution of algorithms *for that specific subject* is always perfectly balanced. For example, if Subject 63 is used for 154 morphs, there are exactly 22 images for each of the 7 algorithms.

## 3 - Face Extraction & Preprocessing

This section processes the raw images to extract the faces, standardize their dimensions, and remove irrelevant background information.
### Processing Pipeline:
* **SSD Face Detection**: We use OpenCV's DNN (ResNet-10 SSD) to detect faces. Detections are cropped with a 15% margin and padded into a square to prevent aspect-ratio distortion.
* **Fallback Method:** If the model fails to confidently detect a face, we automatically apply a static center crop.
* **Standardization**: Every extracted face is resized to **224x224 pixels**, which is the standard input size for most modern vision backbones.
* **Metadata Tracking**: The processed images are saved in a new `FEI_Processed` directory. A new DataFrame (`full_fei_processed_df`) is generated, tracking the new file paths and adding a `detection_type` column (`ssd` or `cc` for center crop) to monitor the extraction quality.

In [ ]:
# CONFIGURATION 
processed_root = "FEI_Processed"
target_size = (224, 224)
confidence_threshold = 0.6

# Load SSD (Face Detector)
prototxt_path = UTILITY_DIR / "FaceDetection/deploy.prototxt"
model_path = UTILITY_DIR / "FaceDetection/res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)

def process_face(image_path, target_size):
    frame = cv2.imread(image_path)
    if frame is None: return None, "error"
    
    (h, w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)), 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()

    processed_frame = None
    method = "cc" 

    # Face Detection via SSD
    if detections.shape[2] > 0:
        i = np.argmax(detections[0, 0, :, 2])
        confidence = detections[0, 0, i, 2]
        
        if confidence > confidence_threshold:
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")
            
            margin = int(max(x2-x1, y2-y1) * 0.2)
            cx1, cy1 = max(0, x1-margin), max(0, y1-margin)
            cx2, cy2 = min(w, x2+margin), min(h, y2+margin)
            
            face = frame[cy1:cy2, cx1:cx2]
            if face.size > 0:
                fh, fw = face.shape[:2]
                side = max(fh, fw)
                square = np.zeros((side, side, 3), np.uint8)
                square[(side-fh)//2:(side-fh)//2+fh, (side-fw)//2:(side-fw)//2+fw] = face
                processed_frame = cv2.resize(square, target_size)
                method = "ssd"

    # Fallback: Center Crop if SSD fails
    if processed_frame is None:
        min_dim = min(h, w)
        start_x, start_y = (w - min_dim) // 2, (h - min_dim) // 2
        crop = frame[start_y:start_y+min_dim, start_x:start_x+min_dim]
        processed_frame = cv2.resize(crop, target_size)
    
    return processed_frame, method

# MAIN LOOP
for label in ["original", "fake"]:
    os.makedirs(os.path.join(processed_root, label), exist_ok=True)

processed_data = []

print(f"--- STARTING FACE EXTRACTION & RESIZE TO {target_size} ---")

for idx, row in tqdm(full_fei_df.iterrows(), total=len(full_fei_df)):
    img_path = row['path']
    
    val_label = str(row['label'])
    if val_label in ['0', 'original']:
        label_folder = "original"
    elif val_label in ['1', 'fake']:
        label_folder = "fake"
    else:
        label_folder = val_label
    
    face_img, detect_method = process_face(img_path, target_size)
    
    if face_img is not None:
        base_name, extension = os.path.splitext(row['filename'])
        new_filename = f"{base_name}_{detect_method}{extension}"
        new_path = os.path.join(processed_root, label_folder, new_filename)
        
        cv2.imwrite(new_path, face_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        
        new_row = row.to_dict()
        new_row['path'] = new_path 
        new_row['filename'] = new_filename
        new_row['detection_type'] = detect_method
        new_row['resolution'] = f"{target_size[0]}x{target_size[1]}"
        processed_data.append(new_row)

full_fei_processed_df = pd.DataFrame(processed_data)

print(f"\n--- EXTRACTION COMPLETE ---")
print(f"Total processed images: {len(full_fei_processed_df)}")
print(f"Processed images saved in: {processed_root}")
full_fei_processed_df.head()

## 4 - Visual Sanity Check
This serves as a critical "sanity check" to verify that the frame extraction process completed successfully without generating corrupted or blank images

In [ ]:
print("--- VISUAL SANITY CHECK (BALANCED) ---")

def visual_check_balanced(df, n_samples=16):
    if len(df) == 0:
        print("ERROR: DataFrame is empty.")
        return

    df_fake = df[df['label'].astype(str).isin(['1', '1.0', 'fake'])]
    df_real = df[df['label'].astype(str).isin(['0', '0.0', 'original', 'real'])]

    half = n_samples // 2
    
    s_fake = df_fake.sample(min(half, len(df_fake)), random_state=42)
    s_real = df_real.sample(min(half, len(df_real)), random_state=42)
    
    samples = pd.concat([s_fake, s_real]).sample(frac=1, random_state=42).reset_index(drop=True)

    rows = 4
    cols = 4
    fig, axes = plt.subplots(rows, cols, figsize=(16, 16))
    fig.suptitle("FEI Processed Samples (Mixed Real/Fake)", fontsize=20, fontweight='bold')
    axes = axes.flatten()

    for i, (idx, row) in enumerate(samples.iterrows()):
        if i >= len(axes): break
        ax = axes[i]
        img_path = row['path']

        current_label = str(row['label'])
        is_fake = current_label in ['1', '1.0', 'fake']
        label_text = "FAKE" if is_fake else "REAL"
        
        method = str(row.get('detection_type', 'Unknown')).upper()
        resolution = row.get('resolution', 'Unknown')
        
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            
            color = 'red' if is_fake else 'green'
            ax.set_title(f"Label: {label_text}\nDetect: {method}\nSize: {resolution}", color=color, fontsize=10)
        except Exception as e:
            ax.text(0.5, 0.5, "Load Error", ha='center', va='center')
            print(f"Errore nel caricare {img_path}: {e}")
        
        ax.axis('off')

    for j in range(len(samples), len(axes)):
        axes[j].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

visual_check_balanced(full_fei_processed_df)

## 5 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_images/fei_images`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [ ]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "fei_images.csv")
full_fei_processed_df.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")